# Daily market and macro dataset

Build a 15-year daily dataset for the clustering / Entropy Pooling research project. This notebook only downloads and aligns observations; clustering comes later. No upstream Fortitudo code is copied here.

Run all cells from the repository folder. In VS Code, select the repository's `.venv` Python kernel. If dependencies are missing, uncomment the installation cell below.

The cutoff is fixed for reproducibility: **2011-10-05 through 2026-10-04**, using available market observations and excluding the cutoff day's potentially incomplete prices. Change `AS_OF` to refresh the research window.

In [1]:
# Uncomment once if needed, then restart the kernel.
# %pip install pandas numpy yfinance requests ipykernel

## Variables and sources

| Column | Definition | Units | Source |
| --- | --- | --- | --- |
| `sp500` | S&P 500 price index, daily close (not total return) | Index points | [Yahoo Finance: ^GSPC](https://finance.yahoo.com/quote/%5EGSPC/history/) |
| `breakeven_10y` | 10-year breakeven inflation rate | Percent | [FRED: T10YIE](https://fred.stlouisfed.org/series/T10YIE) |
| `real_yield_10y` | 10-year inflation-indexed Treasury constant-maturity yield | Percent | [FRED: DFII10](https://fred.stlouisfed.org/series/DFII10) |
| `slope_10y_2y` | 10-year minus 2-year Treasury yield | Percentage points | [FRED: T10Y2Y](https://fred.stlouisfed.org/series/T10Y2Y) |
| `baa_10y_spread` | Moody’s seasoned Baa corporate bond yield minus 10-year Treasury yield | Percentage points | [FRED: BAA10Y](https://fred.stlouisfed.org/series/BAA10Y) |
| `vix` | Cboe VIX, daily close (30-day horizon) | Annualized volatility percent | [Cboe history](https://www.cboe.com/tradable_products/vix/vix_historical_data/) |
| `vix3m` | Cboe VIX3M, daily close (3-month horizon) | Annualized volatility percent | [Cboe VIX3M](https://www.cboe.com/us/indices/dashboard/vix3m/) |
| `vix_term_spread` | **VIX minus VIX3M** | Volatility percentage points | Calculated here |

Rates and volatility are stored in their published units: `2.5` means 2.5%, not 0.025. These are levels, yields and spreads, rather than eight asset prices. FRED's S&P 500 series has only 10 years of history, hence the separate equity source. No API key is needed.

In [2]:
from pathlib import Path
from io import StringIO
import json
import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

AS_OF = pd.Timestamp("2026-10-05")  # exclusive endpoint; edit to refresh
START = AS_OF - pd.DateOffset(years=15)
END = AS_OF - pd.Timedelta(days=1)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
yf.set_tz_cache_location(str(DATA_DIR / "yfinance_cache"))
retrieved_at = pd.Timestamp.now(tz="UTC").isoformat()
print(f"Requested window: {START.date()} to {END.date()}")

Requested window: 2011-10-05 to 2026-10-04


## 1. Download S&P 500 closes

Use the price index itself, not SPY as a proxy. Save the returned source table locally. A failed download stops the notebook rather than silently changing the asset.

In [3]:
equity = yf.download(
    "^GSPC",
    start=START.strftime("%Y-%m-%d"),
    end=AS_OF.strftime("%Y-%m-%d"),  # Yahoo's end is exclusive
    interval="1d",
    auto_adjust=False,
    multi_level_index=False,
    progress=False,
    threads=False,
    timeout=30,
)
if equity is None or equity.empty:
    raise RuntimeError("S&P 500 download failed. Check the connection and rerun this cell.")
equity.index = pd.DatetimeIndex(equity.index).tz_localize(None).normalize()
equity.to_csv(RAW_DIR / "yahoo_sp500.csv", index_label="date")
sp500 = pd.to_numeric(equity["Close"], errors="coerce").dropna().rename("sp500")

## 2. Download Treasury variables, credit spread and volatility indices

FRED uses missing values on some holidays. Keep them as `NaN`. Download VIX and VIX3M directly from Cboe's CSV endpoints.

In [4]:
def download_csv(url, filename, params=None):
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    table = pd.read_csv(StringIO(response.text), na_values=["."])
    (RAW_DIR / filename).write_text(response.text, encoding="utf-8")
    return table

fred_ids = {
    "breakeven_10y": "T10YIE",
    "real_yield_10y": "DFII10",
    "slope_10y_2y": "T10Y2Y",
    "baa_10y_spread": "BAA10Y",
}
series = {"sp500": sp500}
source_urls = {"sp500": "https://finance.yahoo.com/quote/%5EGSPC/history/"}

for name, series_id in fred_ids.items():
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv"
    table = download_csv(url, f"fred_{series_id}.csv", params={
        "id": series_id,
        "cosd": START.strftime("%Y-%m-%d"),
        "coed": END.strftime("%Y-%m-%d"),
    })
    dates = pd.to_datetime(table.iloc[:, 0])
    values = pd.to_numeric(table[series_id], errors="coerce")
    series[name] = pd.Series(values.to_numpy(), index=dates, name=name)
    source_urls[name] = f"https://fred.stlouisfed.org/series/{series_id}"

for name, index_id in {"vix": "VIX", "vix3m": "VIX3M"}.items():
    url = f"https://cdn.cboe.com/api/global/us_indices/daily_prices/{index_id}_History.csv"
    table = download_csv(url, f"cboe_{index_id}.csv")
    table.columns = table.columns.str.strip().str.upper()
    dates = pd.to_datetime(table["DATE"], format="%m/%d/%Y")
    values = pd.to_numeric(table["CLOSE"], errors="coerce")
    series[name] = pd.Series(values.to_numpy(), index=dates, name=name)
    source_urls[name] = url

## 3. Align on S&P 500 trading dates

`daily_levels` retains all downloaded S&P 500 trading dates, including missing macro observations. No forward-fill, backward-fill or interpolation is applied. `daily_complete` keeps only dates with all eight variables observed. This avoids inventing price or yield changes, but excludes some equity trading days when the Treasury market is closed.

If calculating daily equity returns later, calculate them on `daily_levels` **before** filtering complete rows; otherwise a return could span more than one trading day.

In [5]:
for name, values in series.items():
    values = values.sort_index().loc[START:END]
    if values.index.has_duplicates:
        raise ValueError(f"Duplicate dates in {name}")
    observed = values.dropna()
    if observed.empty or observed.index.min() > START + pd.Timedelta(days=7):
        raise ValueError(f"{name} does not cover the beginning of the 15-year window")
    if observed.index.max() < END - pd.Timedelta(days=7):
        warnings.warn(f"{name} ends at {observed.index.max().date()}; check source freshness")
    series[name] = values

all_dates = pd.concat(series.values(), axis=1).sort_index()
daily_levels = all_dates.reindex(series["sp500"].dropna().index).copy()
daily_levels.index.name = "date"
daily_levels["vix_term_spread"] = daily_levels["vix"] - daily_levels["vix3m"]
daily_complete = daily_levels.dropna().copy()

assert not daily_complete.empty
assert daily_levels.index.is_unique and daily_levels.index.is_monotonic_increasing
assert np.isfinite(daily_complete.to_numpy()).all()
assert (daily_complete[["sp500", "vix", "vix3m"]] > 0).all().all()
assert np.allclose(
    daily_complete["vix_term_spread"],
    daily_complete["vix"] - daily_complete["vix3m"],
)

quality = pd.DataFrame({
    "observed_rows": daily_levels.count(),
    "missing_rows": daily_levels.isna().sum(),
    "first_observed": daily_levels.apply(lambda s: s.first_valid_index()),
    "last_observed": daily_levels.apply(lambda s: s.last_valid_index()),
})
print(f"Equity trading dates: {len(daily_levels):,}")
print(f"Complete dates: {len(daily_complete):,}")
print(f"Dates excluded from complete dataset: {len(daily_levels) - len(daily_complete):,}")
display(quality)
display(daily_complete.head())
display(daily_complete.tail())

Equity trading dates: 3,770
Complete dates: 3,739
Dates excluded from complete dataset: 31


/var/folders/gd/46g2hsl52yx0m_gmp4131fdh0000gp/T/ipykernel_10642/3661607310.py:12: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  all_dates = pd.concat(series.values(), axis=1).sort_index()


,observed_rows,missing_rows,first_observed,last_observed
sp500,3770,0,2011-10-05,2026-10-02
breakeven_10y,3742,28,2011-10-05,2026-10-02
real_yield_10y,3741,29,2011-10-05,2026-10-01
slope_10y_2y,3742,28,2011-10-05,2026-10-02
baa_10y_spread,3739,31,2011-10-05,2026-10-01
vix,3770,0,2011-10-05,2026-10-02
vix3m,3770,0,2011-10-05,2026-10-02
vix_term_spread,3770,0,2011-10-05,2026-10-02


,sp500,breakeven_10y,real_yield_10y,slope_10y_2y,baa_10y_spread,vix,vix3m,vix_term_spread
date,,,,,,,,
2011-10-05,1144.030029,1.80,0.12,1.67,3.36,37.81,38.31,-0.50
2011-10-06,1164.969971,1.88,0.13,1.72,3.34,36.27,37.04,-0.77
2011-10-07,1155.459961,1.94,0.16,1.80,3.32,36.20,37.31,-1.11
2011-10-11,1195.540039,1.95,0.23,1.86,3.32,32.86,34.06,-1.20
2011-10-12,1207.250000,1.98,0.26,1.95,3.33,31.26,32.97,-1.71


,sp500,breakeven_10y,real_yield_10y,slope_10y_2y,baa_10y_spread,vix,vix3m,vix_term_spread
date,,,,,,,,
2026-09-25,7743.410156,2.34,2.83,0.36,1.46,14.87,17.93,-3.06
2026-09-28,7683.689941,2.34,2.90,0.32,1.46,16.07,18.23,-2.16
2026-09-29,7670.839844,2.35,2.91,0.37,1.46,16.04,18.09,-2.05
2026-09-30,7651.540039,2.36,2.93,0.41,1.47,16.34,18.37,-2.03
2026-10-01,7666.450195,2.36,2.88,0.46,1.49,16.39,18.58,-2.19


## 4. Save local datasets

The CSV files, raw downloads and metadata stay in the ignored `data/` directory. The notebook is the reproducible recipe; source data retain their providers' terms. These are currently available historical observations, not an archive of what was known on every past trading date. In a predictive experiment, account for source publication times and use features only when available.

In [6]:
daily_levels.to_csv(DATA_DIR / "daily_macro_levels.csv")
daily_complete.to_csv(DATA_DIR / "daily_macro_complete.csv")
quality.to_csv(DATA_DIR / "data_quality.csv")

metadata = {
    "retrieved_at_utc": retrieved_at,
    "as_of_exclusive": AS_OF.date().isoformat(),
    "requested_start": START.date().isoformat(),
    "requested_end": END.date().isoformat(),
    "calendar": "observed S&P 500 trading dates",
    "missing_value_policy": "no filling; separate complete-case dataset",
    "term_spread": "VIX - VIX3M",
    "rows_levels": len(daily_levels),
    "rows_complete": len(daily_complete),
    "first_complete_date": daily_complete.index.min().date().isoformat(),
    "last_complete_date": daily_complete.index.max().date().isoformat(),
    "sources": source_urls,
    "versions": {"pandas": pd.__version__, "numpy": np.__version__, "yfinance": yf.__version__},
}
(DATA_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved datasets and metadata to {DATA_DIR.resolve()}")

Saved datasets and metadata to /Users/lajosgalambos/Desktop/Quantitative_Investment_Management 2/Chapters/clustering-entropy-pooling/data


## Monthly earnings: assessment for a later extension

**A useful candidate for a slow-moving corporate state, provided point-in-time data are available.** Carry forward the latest published earnings value onto each daily date, starting only when it became available to the strategy. Use the release timestamp (or next trading day for an after-close release), not the month the figure describes. Keep the original release date and the age of the observation. Do not backward-fill or interpolate between releases.

A monthly observation repeated across daily rows supplies context, not new daily information. Fifteen years provide roughly 180 monthly updates, so persistent values can influence cluster assignments for many consecutive days. Evaluate their incremental value against the daily-only baseline with chronological validation and account for serial dependence. Revised realized earnings and historical analyst estimates require their original vintages; a latest-history download does not establish what investors knew at the time. [FRED/ALFRED explains the vintage distinction](https://fred.stlouisfed.org/docs/api/fred/realtime_period.html).

For the first clustering experiment, keep the daily-only dataset above. Later, choose a specific earnings measure and source before adding it.

## Notes for the next research step

- Keep these raw levels as the source dataset. Consider equity returns or trailing returns rather than the drifting S&P 500 index level for regime clustering; choose transformations to match the research question.
- VIX, VIX3M and their difference are exactly linearly dependent. Retain all three in the dataset, but avoid using all three as equally weighted clustering inputs without justification: that would repeatedly weight the volatility dimension.
- Scale chosen features using training-period statistics only. Evaluate clusters chronologically before integrating them into the Fortitudo scenario / EP pipeline.